[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/gemma_from_scratch/blob/main/workshop/07_the_transformer_block.ipynb)


# Workshop: Building Gemma 3 from Scratch

---

## Notebook 7: The Transformer Block

**Estimated Time: 20 minutes**

Now we assemble all the pieces! A Transformer block (or layer) is the fundamental repeating unit of the model. In Gemma 3, we alternate between **local** and **global** layers: layers 0-4 are local (sliding window=1024), layer 5 is global (full attention), then repeat.

---

## Learning Objectives:
1. Assemble Attention, GeGLU MLP, and RMSNorm into a single block.
2. Implement the full double-normalization residual connections.
3. Understand how the local/global pattern alternates.
4. Build a complete Gemma 3 transformer block with QK-Norm attention, GQA, GeGLU, and double RMSNorm.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

# Gemma 3 architecture parameters (scaled to ~270M)
hidden_size = 768
num_heads = 8
num_kv_heads = 2
num_key_value_groups = num_heads // num_kv_heads  # = 4
head_dim = hidden_size // num_heads  # = 96
intermediate_size = 2048
sliding_window = 1024
hidden_scale = hidden_size**0.5  # = sqrt(768) ≈ 27.7

Python version: 3.12.13 (main, Mar 10 2026, 18:15:41) [Clang 21.1.4 ]
PyTorch version: 2.5.1


---

## 1. Define All Components

In [2]:
# ─── GQA Layer ───
class Gemma3GQA(nn.Module):
    """Grouped Query Attention with QK-Norm (Gemma 3)."""

    def __init__(self, d_in, n_heads, n_kv_groups, h_dim, is_local=True):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_groups = n_kv_groups
        self.h_dim = h_dim
        self.group_size = n_heads // n_kv_groups
        self.is_local = is_local  # Determines which RoPE freq to use

        self.W_q = nn.Linear(d_in, n_heads * h_dim, bias=False)
        self.W_k = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.W_v = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * h_dim, d_in, bias=False)

    def forward(self, x):
        B, T, C = x.shape

        # 1. Project
        q = self.W_q(x).view(B, T, self.n_heads, self.h_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)

        # 2. Expand K, V groups to match Q heads
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)

        # 3. Apply causal mask if needed (skipped for brevity)

        # 4. QK-Norm attention (Gemma 3 innovation)
        q_norm = F.normalize(q, p=2, dim=-1)
        k_norm = F.normalize(k, p=2, dim=-1)
        scores = torch.matmul(q_norm, k_norm.transpose(-2, -1)) / math.sqrt(self.h_dim)
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)

        # 5. Reshape and project out
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.h_dim)
        return self.out_proj(out)


# ─── GeGLU MLP ───
class Gemma3GeGLU(nn.Module):
    """Gemma 3 GeGLU (GELU-based Gated MLP)."""

    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.gate_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.up_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.down_proj = nn.Linear(d_hidden, d_in, bias=False)

    def forward(self, x):
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        activated_gate = F.gelu(gate, approximate="tanh")
        return self.down_proj(activated_gate * up)


# ─── RMSNorm (Gemma's Add-One) ───
class Gemma3RMSNorm(nn.Module):
    """Gemma 3-style RMSNorm with Add-One initialization."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        return self._norm(x.float()).type_as(x) * (1.0 + self.weight)

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)


print("All components defined successfully!")
print(f"  hidden_size = {hidden_size}")
print(f"  num_heads = {num_heads}")
print(f"  num_kv_heads = {num_kv_heads}")
print(f"  intermediate_size = {intermediate_size}")
print(f"  sliding_window = {sliding_window}")

All components defined successfully!
  hidden_size = 768
  num_heads = 8
  num_kv_heads = 2
  intermediate_size = 2048
  sliding_window = 1024


---

## 🏗️ Assembling the Gemma 3 Block with Double-Normalization

Here we assemble all the components we have built into a complete **Gemma 3 Transformer Block**. 

Gemma 3 features a unique **double-normalization** pattern. Traditional models (like Llama) use pre-normalization, applying one norm per sub-layer (before attention and before MLP). Gemma 3 applies **both pre-normalization and post-normalization** to every sub-layer, using 4 RMSNorms per block.

### 🔄 Data Flow with Double-Normalization:
1. **Attention Sub-layer**:

   $$x_{norm} = \text{RMSNorm}_{pre}(x)$$

   $$x_{attn} = \text{Attention}(x_{norm})$$

   $$x_{post\_attn} = \text{RMSNorm}_{post}(x_{attn})$$

   $$x_{res1} = x + x_{post\_attn} \quad \text{(First Residual)}$$

2. **Gated MLP (GeGLU) Sub-layer**:

   $$x_{norm2} = \text{RMSNorm}_{pre2}(x_{res1})$$

   $$x_{mlp} = \text{GeGLU}(x_{norm2})$$

   $$x_{post\_mlp} = \text{RMSNorm}_{post2}(x_{mlp})$$

   $$\text{Output} = x_{res1} + x_{post\_mlp} \quad \text{(Second Residual)}$$

This double-norm setup provides extreme training stability, allowing deep networks to scale reliably without gradient explosion!

In [3]:
class Gemma3TransformerBlock(nn.Module):
    """Complete Gemma 3 Transformer block with all innovations."""

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_groups,
        h_dim,
        hidden_dim,
        is_local=True,
        sliding_window=None,
    ):
        super().__init__()

        # Sub-layers
        self.attn = Gemma3GQA(dim, n_heads, n_kv_groups, h_dim, is_local=is_local)
        self.ffn = Gemma3GeGLU(dim, hidden_dim)

        # 4 RMSNorm layers (double-norm pattern)
        self.input_layernorm = Gemma3RMSNorm(dim)  # Pre-norm on attention
        self.post_attn_layernorm = Gemma3RMSNorm(dim)  # Post-norm on attention
        self.pre_feedforward_layernorm = Gemma3RMSNorm(dim)  # Pre-norm on MLP
        self.post_feedforward_layernorm = Gemma3RMSNorm(dim)  # Post-norm on MLP

        self.is_local = is_local
        self.sliding_window = sliding_window

    def forward(self, x):
        # === Attention sub-block with double norm ===
        shortcut = x
        x = self.input_layernorm(x)
        x = self.attn(x)
        x = self.post_attn_layernorm(x)
        x = shortcut + x

        # === MLP sub-block with double norm ===
        shortcut = x
        x = self.pre_feedforward_layernorm(x)
        x = self.ffn(x)
        x = self.post_feedforward_layernorm(x)
        x = shortcut + x

        return x

    def __repr__(self):
        att_type = (
            f"Local(sw={self.sliding_window})" if self.is_local else "Global(none)"
        )
        return f"Gemma3Block({att_type})"


# Instantiate a local and global block
local_block = Gemma3TransformerBlock(
    hidden_size,
    num_heads,
    num_kv_heads,
    head_dim,
    intermediate_size,
    is_local=True,
    sliding_window=sliding_window,
)
global_block = Gemma3TransformerBlock(
    hidden_size, num_heads, num_kv_heads, head_dim, intermediate_size, is_local=False
)

print(f"Local block: {local_block}")
print(f"Global block: {global_block}")

Local block: Gemma3Block(Local(sw=1024))
Global block: Gemma3Block(Global(none))


---

## 3. Data Flow Visualization

In [4]:
# Test the block with dummy input
x = torch.randn(1, 8, hidden_size)
print(f"Input shape:  {x.shape}")

# Run through local + global blocks
x_local = local_block(x)
x_global = global_block(x_local)

print(f"After local:  {x_local.shape}")
print(f"After global: {x_global.shape}")
assert x_local.shape == x.shape
assert x_global.shape == x.shape
print("\n✅ Blocks preserve shape correctly!")

Input shape:  torch.Size([1, 8, 768])
After local:  torch.Size([1, 8, 768])
After global: torch.Size([1, 8, 768])

✅ Blocks preserve shape correctly!


---

## 4. Parameter Count per Block

In [5]:
def count_params(block):
    return sum(p.numel() for p in block.parameters())


local_params = count_params(local_block)
print(f"Local block parameters: {local_params:,}")
print(f"  Attention: {count_params(local_block.attn):,}")
print(f"  GeGLU MLP: {count_params(local_block.ffn):,}")
print(f"  RMSNorm: {count_params(local_block.input_layernorm):,} x 4")

# Compare to Gemma 3 spec: ~270M total
n_layers = 8
total_est = n_layers * local_params + 2 * hidden_size * 256000
print(f"\n~270M estimated total with {n_layers} blocks + vocab: {total_est:,}")

Local block parameters: 6,196,224
  Attention: 1,474,560
  GeGLU MLP: 4,718,592
  RMSNorm: 768 x 4

~270M estimated total with 8 blocks + vocab: 442,785,792


---

## 5. Full Block Pattern Visualization

In [6]:
# Show the layer pattern for 8 layers in our ~270M model
print("Layer Pattern (5 local + 1 global, repeating):")
for layer_idx in range(n_layers):
    is_global = layer_idx % 6 == 5
    layer_type = "GLOBAL" if is_global else "LOCAL"
    window = "n/a" if is_global else str(sliding_window)
    rope_freq = "1M (x8)" if is_global else "10K"

    bar = "\u2588" * 4 if is_global else "\u2588" * 1
    print(
        f"  Layer {layer_idx:2d}: {layer_type:6s} |{bar:<4s}| "
        f"sliding={window:>4s}  RoPE={rope_freq}"
    )

print("\nEvery 6th layer (0-indexed) is GLOBAL -- full attention.")
print("Other 5 layers are LOCAL -- attend only to recent sliding window.")

Layer Pattern (5 local + 1 global, repeating):
  Layer  0: LOCAL  |█   | sliding=1024  RoPE=10K
  Layer  1: LOCAL  |█   | sliding=1024  RoPE=10K
  Layer  2: LOCAL  |█   | sliding=1024  RoPE=10K
  Layer  3: LOCAL  |█   | sliding=1024  RoPE=10K
  Layer  4: LOCAL  |█   | sliding=1024  RoPE=10K
  Layer  5: GLOBAL |████| sliding= n/a  RoPE=1M (x8)
  Layer  6: LOCAL  |█   | sliding=1024  RoPE=10K
  Layer  7: LOCAL  |█   | sliding=1024  RoPE=10K

Every 6th layer (0-indexed) is GLOBAL -- full attention.
Other 5 layers are LOCAL -- attend only to recent sliding window.


---

## Key Takeaway

A Gemma 3 block has:
1. **GQA** with QK-Norm (not soft-capping)
2. **GeGLU** (not SwiGLU)
3. **4 RMSNorm** layers: pre + post on both attention and MLP
4. **Double-residual**: each sub-layer adds back its pre-sub-layer output

---

[<- Previous Notebook (06_rmsnorm_and_normalization.ipynb)](06_rmsnorm_and_normalization.ipynb) | [Next Notebook (08_gemma_model_assembly.ipynb) ->](08_gemma_model_assembly.ipynb)
